# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 9.0 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
TASK_ID = "task206"
CH = 10
H = W = 30
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")
LOCAL_TASK_JSON = Path("/mnt/data/task206.json")
KAGGLE_TASK_JSON = Path(COMPETITION) / "task206.json"
TASK_JSON = LOCAL_TASK_JSON if LOCAL_TASK_JSON.exists() else KAGGLE_TASK_JSON
OUT_DIR = WORK_DIR / "task206_marker_motif_copy_onnx"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUBMISSION_PATH = WORK_DIR / "submission.zip"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_validation_summary.json"
with TASK_JSON.open("r") as f:
    task = json.load(f)
print(TASK_ID, len(task.get("train", [])), len(task.get("test", [])), len(task.get("arc-gen", [])))

task206 3 1 262


In [6]:

def grid_to_tensor(grid, h=H, w=W, ch=CH):
    """Encode an ARC grid as [1,10,30,30].

    Grid cells, including color-0 background inside the true grid, are one-hot.
    Padding outside the true grid is all-zero across channels, so active_canvas is
    exactly sum(input_channels) > 0.
    """
    x=np.zeros((1,ch,h,w),dtype=np.float32)
    for r,row in enumerate(grid):
        for c,v in enumerate(row):
            x[0,int(v),r,c]=1.0
    return x

class Task206MarkerMotifCopy(nn.Module):
    """Static symbolic model for marker-guided motif copying.

    Rule:
      1. active_canvas = sum(input_channels) > 0.
      2. color 5 is the marker, not ordinary background.
      3. foreground motif = nonzero colors excluding marker 5.
      4. compute motif bounding-box center, align it to marker position.
      5. copy motif by that translation; remove marker; clip by active_canvas.

    No visible output template is stored. The copy uses a dynamic translation
    inferred from the input and a static 30x30 sampling grid.
    """
    def __init__(self):
        super().__init__()
        rr=torch.arange(H,dtype=torch.float32).view(1,H,1).expand(1,H,W)
        cc=torch.arange(W,dtype=torch.float32).view(1,1,W).expand(1,H,W)
        self.register_buffer("rr", rr)
        self.register_buffer("cc", cc)

    def forward(self, x):
        active=(x.sum(dim=1, keepdim=True)>0.5).float()
        marker=x[:,5:6]

        # Motif support: nonzero foreground, excluding the color-5 marker.
        non_bg=x[:,1:10].sum(dim=1, keepdim=True)
        src_mask=((non_bg-marker)>0.5).float()*active

        # Source motif bounding-box center, computed without NonZero/Unique/Loop.
        row_presence=(src_mask.amax(dim=3)>0.5).float()
        col_presence=(src_mask.amax(dim=2)>0.5).float()
        min_r=torch.argmax(row_presence, dim=2).to(torch.float32).view(1,1,1)
        max_r=(float(H-1)-torch.argmax(torch.flip(row_presence,dims=[2]), dim=2).to(torch.float32)).view(1,1,1)
        min_c=torch.argmax(col_presence, dim=2).to(torch.float32).view(1,1,1)
        max_c=(float(W-1)-torch.argmax(torch.flip(col_presence,dims=[2]), dim=2).to(torch.float32)).view(1,1,1)
        center_r=torch.floor((min_r+max_r)/2.0)
        center_c=torch.floor((min_c+max_c)/2.0)

        # Marker coordinate.
        mflat=marker.reshape(1,-1)
        midx=torch.argmax(mflat, dim=1).to(torch.float32).view(1,1,1)
        mr=torch.floor(midx/float(W))
        mc=midx-mr*float(W)

        # For every output pixel, sample the motif from the translated source.
        src_r=self.rr-(mr-center_r)
        src_c=self.cc-(mc-center_c)
        gy=2.0*src_r/float(H-1)-1.0
        gx=2.0*src_c/float(W-1)-1.0
        grid=torch.stack([gx,gy], dim=-1)

        src_fore=x*src_mask
        copied=F.grid_sample(src_fore, grid, mode="nearest", padding_mode="zeros", align_corners=True)

        # Preserve original foreground except the marker, overlay copied motif,
        # reconstruct channel 0 as true background only inside active_canvas.
        orig_fore=x[:,1:10]*((1.0-marker)*active)
        copied_fore=copied[:,1:10]*active
        fg=torch.clamp(orig_fore+copied_fore,0.0,1.0)*active
        occ=torch.clamp(fg.sum(dim=1, keepdim=True),0.0,1.0)
        bg=active*(1.0-occ)
        return torch.cat([bg,fg], dim=1)*active

model = Task206MarkerMotifCopy().eval()

# Symbolic sanity check before export.
with torch.no_grad():
    for split in ["train", "test", "arc-gen"]:
        ok=0; bad=[]
        for i,ex in enumerate(task.get(split, [])):
            y=model(torch.from_numpy(grid_to_tensor(ex["input"]))).numpy()
            exp=grid_to_tensor(ex["output"])
            if np.array_equal((y>0.5).astype(np.float32), exp):
                ok += 1
            else:
                bad.append(i)
        print(split, ok, "/", len(task.get(split, [])), "bad", bad[:10])


train 3 / 3 bad []
test 1 / 1 bad []
arc-gen 262 / 262 bad []


In [7]:
dummy = torch.from_numpy(grid_to_tensor(task["test"][0]["input"]))
torch.onnx.export(
    model, dummy, str(ONNX_PATH), input_names=["input"], output_names=["output"],
    opset_version=17, do_constant_folding=True, dynamic_axes=None, dynamo=False,
)
onnx_model = onnx.load(str(ONNX_PATH))
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx.save(onnx_model, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH))
ONNX_PATH, ONNX_PATH.stat().st_size

/tmp/ipykernel_16/1648413716.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


(PosixPath('/kaggle/working/task206_marker_motif_copy_onnx/task206.onnx'),
 22326)

In [8]:
def vi_shape(vi):
    return [int(d.dim_value) if d.dim_value else (d.dim_param or None) for d in vi.type.tensor_type.shape.dim]
onnx_model = onnx.load(str(ONNX_PATH))
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
summary_static = {
    "input_shape": vi_shape(onnx_model.graph.input[0]),
    "output_shape": vi_shape(onnx_model.graph.output[0]),
    "onnx_size_bytes": ONNX_PATH.stat().st_size,
    "ops": dict(ops),
    "forbidden_ops": sorted(forbidden & set(ops)),
    "function_count": len(onnx_model.functions),
}
print(summary_static)
assert summary_static["input_shape"] == [1,10,30,30]
assert summary_static["output_shape"] == [1,10,30,30]
assert summary_static["onnx_size_bytes"] < 1_400_000
assert not summary_static["forbidden_ops"]
assert summary_static["function_count"] == 0

{'input_shape': [1, 10, 30, 30], 'output_shape': [1, 10, 30, 30], 'onnx_size_bytes': 22326, 'ops': {'Constant': 51, 'ReduceSum': 3, 'Greater': 4, 'Cast': 9, 'Slice': 5, 'Sub': 12, 'Mul': 11, 'ReduceMax': 2, 'ArgMax': 5, 'Reshape': 6, 'Add': 3, 'Div': 5, 'Floor': 3, 'Unsqueeze': 2, 'Concat': 2, 'GridSample': 1, 'Clip': 2}, 'forbidden_ops': [], 'function_count': 0}


In [9]:

sess_options = ort.SessionOptions(); sess_options.intra_op_num_threads=1; sess_options.inter_op_num_threads=1
sess = ort.InferenceSession(str(ONNX_PATH), sess_options=sess_options, providers=["CPUExecutionProvider"])

def validate_examples(examples):
    ok=0; bad=[]; outside_zero_ok=0; active_canvas_covered_ok=0
    for i,ex in enumerate(examples):
        x=grid_to_tensor(ex["input"])
        y=sess.run(None,{"input":x})[0]
        pred=(y>0.5).astype(np.float32)
        exp=grid_to_tensor(ex["output"])
        if np.array_equal(pred,exp):
            ok += 1
        else:
            bad.append(i)
        active=(x.sum(axis=1,keepdims=True)>0.5).astype(np.float32)
        outside_zero_ok += bool(np.all(pred*(1-active)==0))
        active_canvas_covered_ok += bool(np.all((pred.sum(axis=1,keepdims=True)>0.5)==active))
    return {"ok":ok,"total":len(examples),"bad_first10":bad[:10],
            "outside_zero_ok":outside_zero_ok,
            "active_canvas_covered_ok":active_canvas_covered_ok}

rng=random.Random(0)
inds=list(range(len(task.get("arc-gen",[]))))
rng.shuffle(inds)
hold=[task["arc-gen"][i] for i in inds[:max(1, int(round(0.60*len(inds))))]] if inds else []
summary = {
    **summary_static,
    "train": validate_examples(task.get("train", [])),
    "test": validate_examples(task.get("test", [])),
    "arc_gen_60pct_holdout": validate_examples(hold),
    "arc_gen_all": validate_examples(task.get("arc-gen", [])),
}
with SUMMARY_PATH.open("w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
assert summary["train"]["ok"] == summary["train"]["total"]
assert summary["test"]["ok"] == summary["test"]["total"]
assert summary["arc_gen_60pct_holdout"]["ok"] == summary["arc_gen_60pct_holdout"]["total"]


{
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 22326,
  "ops": {
    "Constant": 51,
    "ReduceSum": 3,
    "Greater": 4,
    "Cast": 9,
    "Slice": 5,
    "Sub": 12,
    "Mul": 11,
    "ReduceMax": 2,
    "ArgMax": 5,
    "Reshape": 6,
    "Add": 3,
    "Div": 5,
    "Floor": 3,
    "Unsqueeze": 2,
    "Concat": 2,
    "GridSample": 1,
    "Clip": 2
  },
  "forbidden_ops": [],
  "function_count": 0,
  "train": {
    "ok": 3,
    "total": 3,
    "bad_first10": [],
    "outside_zero_ok": 3,
    "active_canvas_covered_ok": 3
  },
  "test": {
    "ok": 1,
    "total": 1,
    "bad_first10": [],
    "outside_zero_ok": 1,
    "active_canvas_covered_ok": 1
  },
  "arc_gen_60pct_holdout": {
    "ok": 157,
    "total": 157,
    "bad_first10": [],
    "outside_zero_ok": 157,
    "active_canvas_covered_ok": 157
  },
  "arc_gen_all": {
    "ok": 262,
    "total": 262,
    "bad_first10": [],
    "outside_zero_ok"

In [10]:

with zipfile.ZipFile(SUBMISSION_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f'{TASK_ID}.onnx')
print('Wrote:', SUBMISSION_PATH)
print('Zip contents:', zipfile.ZipFile(SUBMISSION_PATH).namelist())
assert zipfile.ZipFile(SUBMISSION_PATH).namelist() == [f'{TASK_ID}.onnx']


Wrote: /kaggle/working/submission.zip
Zip contents: ['task206.onnx']
